# 05 · Two-year forecast: test, diagnostics, forward forecast (Team 10, Deliverable 5)

1. Read the model D4 carried (`decision.json`); nothing is re-chosen here.
2. **Test:** forecast the last 24 months of data once from the test origin and score the
   carried model against the seasonal naive. If the naive itself was carried, it is compared
   with the other candidates for context only.
3. **Uncertainty:** an 80% band for the network total per month, calibrated on the stability
   origin (its window ends before the test window), then checked on the test window.
4. **Diagnostics:** error by horizon, by route size and by market, and bias by year.
5. **Forward forecast:** refit on all data and forecast the 24 months after the last month.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODELING = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "team-10-modeling"]
                if (p / "src" / "forecasting.py").exists())
sys.path.insert(0, str(MODELING / "src"))

from data import BANNER, FIGS, TABLES, load_airports, load_pax, tag, write_table
from features import airport_table, clean_pax, route_table
import forecasting as F

print(BANNER)
pax = clean_pax(load_pax())
routes = route_table(pax, airport_table(load_airports()))
o = F.origins(pax)
decision_path = TABLES / "decision.json"
assert decision_path.exists(), "Run 04_model_comparison.ipynb first."
carried = json.loads(decision_path.read_text(encoding="utf-8"))["carried"]
print("carried model:", carried, "| test origin:", o.test.date())

## 1. Test window: one shot from the test origin

In [ ]:
fs_test = F.forecast_set(pax, o.test)
scored, summaries = {}, {}
for name in F.CANDIDATES:
    scored[name] = F.scored_frame(pax, fs_test, F.fit_predict(name, pax, routes, fs_test, o.test), o.test)
    summaries[name] = F.summary(scored[name])
test_table = pd.DataFrame(summaries).T
test_table["role"] = ["carried" if n == carried else ("baseline" if n == "Seasonal naive" else "context only")
                      for n in test_table.index]
test_table.round(4)

## 2. An 80% band for the monthly network total

Calibration uses the **select origin's first 12 months** (they end the month before the test
window, so no test month is used) and the log ratio actual ÷ forecast of the network total. Its
10th and 90th percentiles give the year-1 band. Year 2 has no clean calibration months before the
test window, so its log band is the year-1 band widened by √2 (errors that accumulate like a
random walk over a second year). The test window then checks both.

In [ ]:
fs_sel = F.forecast_set(pax, o.select)
cal = F.scored_frame(pax, fs_sel, F.fit_predict(carried, pax, routes, fs_sel, o.select), o.select)
cal = cal[cal["DATE"] < o.test]
assert cal["h_year"].eq(1).all() and cal["DATE"].max() < o.test
cal_net = cal.groupby("DATE")[["OD_PAX", "forecast"]].sum()
q_lo, q_hi = np.log(cal_net["OD_PAX"] / cal_net["forecast"]).quantile([0.1, 0.9])
band = pd.DataFrame({"low": [np.exp(q_lo), np.exp(q_lo * np.sqrt(2))],
                     "high": [np.exp(q_hi), np.exp(q_hi * np.sqrt(2))]},
                    index=pd.Index([1, 2], name="h_year"))
print(band.round(3))

net = scored[carried].groupby(["DATE", "h_year"])[["OD_PAX", "forecast"]].sum().reset_index()
net = net.merge(band, left_on="h_year", right_index=True)
net["lo"], net["hi"] = net["forecast"] * net["low"], net["forecast"] * net["high"]
coverage = float(((net["OD_PAX"] >= net["lo"]) & (net["OD_PAX"] <= net["hi"])).mean())
cov_by_year = net.assign(inside=(net["OD_PAX"] >= net["lo"]) & (net["OD_PAX"] <= net["hi"])).groupby("h_year")["inside"].mean()
print(f"80% band coverage on the test window: {coverage:.0%} (year 1 {cov_by_year[1]:.0%}, year 2 {cov_by_year[2]:.0%})")

fig, ax = plt.subplots(figsize=(12, 4.5))
hist = pax[pax["DATE"] < o.test].groupby("DATE")["OD_PAX"].sum()
ax.plot(hist.index[-24:], hist.values[-24:], color="grey", label="history")
ax.plot(net["DATE"], net["OD_PAX"], color="black", lw=2, label="actual (test window)")
ax.plot(net["DATE"], net["forecast"], color="tab:blue", label=f"{carried} forecast")
ax.fill_between(net["DATE"], net["lo"], net["hi"], color="tab:blue", alpha=0.15, label="80% band")
ax.axvline(o.test, color="k", ls="--", lw=1)
ax.set(ylabel="passengers per month", title=tag("Test window: observed vs two-year forecast (network total)"))
ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "d5_observed_vs_forecast.png", dpi=120); plt.show()

**NDA-safe version for the repo.** The chart above shows real passenger volumes, so it stays in the
git-ignored output folder. The version below is rescaled so that the average month in the year
before the test origin = 100. It shows the same shape, bias and band, with no passenger counts.

In [ ]:
base = hist.iloc[-12:].mean()
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(hist.index[-24:], 100 * hist.values[-24:] / base, color="grey", label="history")
ax.plot(net["DATE"], 100 * net["OD_PAX"] / base, color="black", lw=2, label="actual (test window)")
ax.plot(net["DATE"], 100 * net["forecast"] / base, color="tab:blue", label=f"{carried} forecast")
ax.fill_between(net["DATE"], 100 * net["lo"] / base, 100 * net["hi"] / base, color="tab:blue", alpha=0.15, label="80% band")
ax.axvline(o.test, color="k", ls="--", lw=1)
ax.set(ylabel="index (avg month in year before test = 100)",
       title=tag("Test window: observed vs two-year forecast (indexed, no passenger counts)"))
ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "d5_observed_vs_forecast_indexed.png", dpi=120); plt.show()

## 3. Diagnostics

In [ ]:
hz = pd.DataFrame({n: F.by_horizon(s) for n, s in scored.items()})
fig, ax = plt.subplots(figsize=(9, 4.5))
for n in hz:
    ax.plot(hz.index, 100 * hz[n], marker="o", ms=3, lw=2.5 if n == carried else 1, label=n)
ax.axvline(12.5, color="k", ls=":", lw=0.8)
ax.set(xlabel="months after origin", ylabel="route-month WAPE (%)", title=tag("Test window: error by horizon"))
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGS / "d5_error_by_horizon.png", dpi=120); plt.show()

seg = F.by_segment(scored[carried], pax, routes, o.test)
seg_base = F.by_segment(scored["Seasonal naive"], pax, routes, o.test).rename(columns={"WAPE": "WAPE_naive"})
seg = seg.merge(seg_base[["segment", "level", "year", "WAPE_naive"]], on=["segment", "level", "year"])
seg.round(4)

In [ ]:
pivot = seg.pivot_table(index=["segment", "level"], columns="year", values="WAPE")
fig, ax = plt.subplots(figsize=(9, 4.5))
(100 * pivot).plot.barh(ax=ax)
ax.set(xlabel="route-month WAPE (%)", ylabel="", title=tag(f"Test window: {carried} error by segment"))
ax.legend(title="forecast year")
fig.tight_layout(); fig.savefig(FIGS / "d5_error_by_segment.png", dpi=120); plt.show()

s = scored[carried]
both = s[(s["OD_PAX"] > 0) & (s["forecast"] > 0)]
log_res = np.log(both["OD_PAX"] / both["forecast"])
print(f"log residual (actual/forecast), matched route-months: median {log_res.median():+.3f}, "
      f"share under-forecast {float((log_res > 0).mean()):.0%}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(log_res.clip(-2, 2), bins=80)
ax.axvline(0, color="k")
ax.set(xlabel="log(actual / forecast), clipped at ±2", ylabel="route-months", title=tag("Test window: residuals"))
fig.tight_layout(); fig.savefig(FIGS / "d5_residuals.png", dpi=120); plt.show()

## 4. Forward forecast: the next 24 months

Refit on everything and forecast from the month after the last observation. There are no actuals
to score; the band reuses the multipliers from section 2.

In [ ]:
fs_fwd = F.forecast_set(pax, o.forward)
fwd = fs_fwd.assign(forecast=F.fit_predict(carried, pax, routes, fs_fwd, o.forward))
fwd_net = fwd.groupby(["DATE", "h_year"])["forecast"].sum().reset_index().merge(band, left_on="h_year", right_index=True)
fwd_net["lo"], fwd_net["hi"] = fwd_net["forecast"] * fwd_net["low"], fwd_net["forecast"] * fwd_net["high"]
last12 = pax[pax["DATE"] >= o.forward - pd.DateOffset(months=12)]["OD_PAX"].sum()
y1, y2 = fwd_net.loc[fwd_net["h_year"] == 1, "forecast"].sum(), fwd_net.loc[fwd_net["h_year"] == 2, "forecast"].sum()
print(f"forecast year 1 vs last 12 months: {y1 / last12 - 1:+.1%} | year 2 vs year 1: {y2 / y1 - 1:+.1%}")

fig, ax = plt.subplots(figsize=(12, 4.5))
h = pax.groupby("DATE")["OD_PAX"].sum()
ax.plot(h.index[-36:], h.values[-36:], color="black", label="observed")
ax.plot(fwd_net["DATE"], fwd_net["forecast"], color="tab:blue", label=f"{carried} forecast")
ax.fill_between(fwd_net["DATE"], fwd_net["lo"], fwd_net["hi"], color="tab:blue", alpha=0.15, label="80% band")
ax.axvline(o.forward, color="k", ls="--", lw=1)
ax.set(ylabel="passengers per month", title=tag("Forward two-year forecast (network total)"))
ax.legend()
fig.tight_layout(); fig.savefig(FIGS / "d5_forward_forecast.png", dpi=120); plt.show()

In [ ]:
def pct(x):
    return f"{100 * x:.2f}%"

L = [f"# Forecast evaluation — Team 10 (D5)\n\n_Generated by `notebooks/05_two_year_forecast.ipynb`._\n",
     f"- Carried model (from D4): **{carried}**",
     f"- Test origin {o.test.date()}: one 24-month forecast, scored against the actual rows, zero-filled",
     f"- Band calibrated on the first 12 months from the select origin ({o.select.date()}), all before the test window\n",
     "## Test window", "",
     "| model | role | WAPE 24m | WAPE y1 | WAPE y2 | bias y1 | bias y2 | network WAPE |", "|---|---|---|---|---|---|---|---|"]
for n, r in test_table.iterrows():
    L.append(f"| {n} | {r.role} | {pct(r.WAPE_24m)} | {pct(r.WAPE_y1)} | {pct(r.WAPE_y2)} | "
             f"{100 * r.bias_y1:+.2f}% | {100 * r.bias_y2:+.2f}% | {pct(r.network_WAPE)} |")
L += ["", "## 80% network band", "", "| forecast year | low multiplier | high multiplier | test coverage |", "|---|---|---|---|"]
L += [f"| {k} | {band.loc[k, 'low']:.3f} | {band.loc[k, 'high']:.3f} | {cov_by_year[k]:.0%} |" for k in (1, 2)]
L += ["", "## Error by segment (carried model vs seasonal naive)", "",
      "| segment | level | year | share of passengers | WAPE | WAPE naive |", "|---|---|---|---|---|---|"]
L += [f"| {r.segment} | {r.level} | {r.year} | {r.share_of_pax:.1%} | {pct(r.WAPE)} | {pct(r.WAPE_naive)} |" for r in seg.itertuples()]
L += ["", "## Route-month WAPE by horizon", "", "| h | " + " | ".join(hz.columns) + " |", "|---" * (len(hz.columns) + 1) + "|"]
L += [f"| {h} | " + " | ".join(pct(v) for v in row) + " |" for h, row in hz.iterrows()]
L += ["", "## Forward forecast", "",
      f"Year 1 vs last 12 observed months: {y1 / last12 - 1:+.1%}; year 2 vs year 1: {y2 / y1 - 1:+.1%}."]
print("wrote", write_table("forecast_evaluation_scores.md", "\n".join(L) + "\n"))

In [ ]:
# Machine-readable copy for scripts/build_deliverables.py (fills the reports and the slides).
_ = (TABLES / "d5_scores.json").write_text(json.dumps({
    "carried": carried,
    "origins": {k: str(v.date()) for k, v in o.__dict__.items()},
    "test": summaries,
    "band": {str(k): {"low": float(band.loc[k, "low"]), "high": float(band.loc[k, "high"])} for k in (1, 2)},
    "coverage": {"all": coverage, "y1": float(cov_by_year[1]), "y2": float(cov_by_year[2])},
    "by_horizon": {n: [float(x) for x in hz[n].values] for n in hz},
    "months_carried_beats_naive": int((hz[carried] < hz["Seasonal naive"]).sum()),
    "segments": seg.to_dict(orient="records"),
    "residual": {"median_log": float(log_res.median()), "share_under": float((log_res > 0).mean())},
    "forward": {"y1_vs_last12": float(y1 / last12 - 1), "y2_vs_y1": float(y2 / y1 - 1)},
}, indent=2, default=float), encoding="utf-8")
print("wrote", TABLES / "d5_scores.json")